# 04c — Multi-scale temporal features

Produces the fourth feature CSV alongside the standard, extended, and normalized variants. Each row corresponds to the same 5-second windows used elsewhere in the pipeline, but the features are computed over a wider 15-second context window centered on each 5-second slot.

**Why this exists.** Bus, metro, and tram dynamics play out over time scales longer than 5 seconds. Bus stop-go cycles last 10 to 20 seconds. Metro transit between stations is smooth across 30 to 60 seconds with discrete events at the boundaries. Tram features regular rail joint impacts every few seconds. A 5-second window truncates most of these patterns. A 15-second context window captures them.

The features target the bus, metro, tram three-way classification at Level 3B of the hierarchical model. They are not useful at Level 1, 2A, or 2B and are not intended to be combined with the other feature sets there.

**Output.** `data/processed/features_common_no_baro_multiscale.csv`. Same rows and metadata as the standard CSV. Feature columns are prefixed `ms_` so the file can be merged with the standard CSV on `(session_id, window_index)` without column collisions.

## 1. Imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features as F
from src.data_loading import build_session_index

INTERIM_DIR   = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Iterate sessions and compute multi-scale features

Same windowing as the standard CSV (5 seconds, 50 percent overlap, 250 samples at 50 Hz). For each window we call `F.compute_multiscale_features(df, s, e)` which builds the 15-second context window centered on the 5-second slot and returns the 19 multi-scale features.

In [2]:
session_index = build_session_index(INTERIM_DIR)
print(f'Found {len(session_index)} sessions')

all_rows = []
skipped = 0

for sess in session_index:
    # Skip non-session files (preprocessing logs, etc.)
    if sess['mode'] not in F.MODES:
        skipped += 1
        continue

    df = pd.read_csv(sess['path']).reset_index(drop=True)
    has_bad = 'bad_sample' in df.columns
    has_flag = 'flagged_sample' in df.columns

    for w_idx, (s, e) in enumerate(F.iterate_windows(len(df))):
        # Same bad_sample guard as the standard pipeline
        if has_bad and bool(df.iloc[s:e]['bad_sample'].any()):
            continue
        try:
            ms_feats = F.compute_multiscale_features(df, s, e)
        except Exception as exc:
            print(f'  ERROR on {sess["session_id"]} window {w_idx}: {exc}')
            continue

        flagged_frac = float(df.iloc[s:e]['flagged_sample'].mean()) if has_flag else 0.0

        all_rows.append({
            'session_id':     sess['session_id'],
            'participant':    sess['participant'],
            'mode':           sess['mode'],
            'mode_id':        F.MODE_TO_ID[sess['mode']],
            'day':            sess['day'],
            'window_index':   w_idx,
            'window_start_s': float(s / F.SAMPLING_RATE),
            'window_end_s':   float(e / F.SAMPLING_RATE),
            'flagged_frac':   flagged_frac,
            **ms_feats,
        })

features_ms_df = pd.DataFrame(all_rows)
print(f'Total windows: {len(features_ms_df)}')
print(f'Sessions skipped (non-session files): {skipped}')
print(f'Total columns: {len(features_ms_df.columns)}')

Found 56 sessions
Total windows: 14513
Sessions skipped (non-session files): 2
Total columns: 28


## 3. Clean up NaN and inf, save the CSV

In [3]:
meta_cols = ['session_id', 'participant', 'mode', 'mode_id', 'day',
             'window_index', 'window_start_s', 'window_end_s', 'flagged_frac']
ms_feature_cols = [c for c in features_ms_df.columns if c not in meta_cols]
print(f'Multi-scale feature columns: {len(ms_feature_cols)}')
print(f'Sample feature names: {ms_feature_cols[:5]}')

nan_count = features_ms_df[ms_feature_cols].isna().sum().sum()
inf_count = int(np.isinf(features_ms_df[ms_feature_cols].values).sum())
if nan_count > 0 or inf_count > 0:
    print(f'Replacing {nan_count} NaN and {inf_count} inf with 0')
    features_ms_df[ms_feature_cols] = (
        features_ms_df[ms_feature_cols].fillna(0.0)
                                        .replace([np.inf, -np.inf], 0.0)
    )

features_ms_df = features_ms_df[meta_cols + ms_feature_cols]
out_path = PROCESSED_DIR / 'features_common_no_baro_multiscale.csv'
features_ms_df.to_csv(out_path, index=False)
print(f'Saved {len(features_ms_df)} rows x {len(features_ms_df.columns)} cols to {out_path}')

Multi-scale feature columns: 19
Sample feature names: ['ms_linacc_mag_mean', 'ms_linacc_mag_std', 'ms_linacc_mag_iqr', 'ms_linacc_mag_p95', 'ms_linacc_mag_range']
Saved 14513 rows x 28 cols to C:\Projects\this project-transport-mode\data\processed\features_common_no_baro_multiscale.csv


## 4. Sanity check the multi-scale features

Verify that the features behave as expected per mode. Key predictions:
- Bus should have the highest `ms_burst_count` (irregular road bumps and stops).
- Tram should have lower `ms_burst_interval_std_s` than bus (rail joints arrive at more regular spacing).
- Metro should have the lowest `ms_burst_count` (smooth between stations).
- Stationary should have `ms_active_frac` near zero.

In [4]:
summary = features_ms_df.groupby('mode')[[
    'ms_burst_count', 'ms_burst_interval_std_s', 'ms_active_frac',
    'ms_linacc_mag_std', 'ms_jerk_linacc_std',
]].median().reindex(F.MODES)

print('Median multi-scale features per mode:')
print(summary.round(3).to_string())

Median multi-scale features per mode:
            ms_burst_count  ms_burst_interval_std_s  ms_active_frac  ms_linacc_mag_std  ms_jerk_linacc_std
mode                                                                                                      
stationary             0.0                    0.000           0.000              0.022               0.834
walking                3.0                    0.000           0.997              1.633              58.924
cycling               36.0                    0.428           0.927              1.002              31.058
metro                  8.0                    0.283           0.040              0.135               3.593
bus                   33.0                    0.269           0.419              0.416              11.684
tram                  12.0                    0.417           0.088              0.172               4.006


## 5. Summary

Four feature CSVs now exist in `data/processed/`:

| File | Features | Used by |
|---|---|---|
| `features_common_no_baro.csv` | Standard (129) | L1, L2B, L3B base |
| `features_common_no_baro_normalized.csv` | Standard on normalized signals (129, prefix `norm_`) | L2A |
| `features_common_no_baro_extended.csv` | Standard + 12 event features | L2B |
| `features_common_no_baro_multiscale.csv` | 19 multi-scale features (prefix `ms_`) | L3B add |

All four share the same row count and metadata, so they can be merged on `(session_id, window_index)` to combine representations at any level. Notebook 08 will load the appropriate file(s) for each classifier in the hierarchical cascade.